# IoT Hub Monitor — обучение тяжёлых ML-моделей в Google Colab (Фаза 4)

Пайплайн: **БД → CSV → Colab GPU (обучение) → веса → мак/Render (инференс)**.

Этот ноутбук **импортирует те же `ml/`-модули, что и прод** (через `git clone` публичной
репы) — он НЕ дублирует логику моделей и persistence. Обученный артефакт
(`<key>.manifest.json` + веса) сохраняется тем же `ModelPersistenceMixin.save`, что и локально.

**Что нужно заранее:** CSV, выгруженный командой
`python manage.py export_training_data --device TEMP-001 --metric temperature --output train_TEMP-001.csv`
(для цикла 4c — по одному CSV на каждое из 4 устройств).

**Про GPU и воспроизводимость.** Учим на **GPU (T4)** — для LSTM это ~5× быстрее CPU.
GPU и CPU дают чуть разные веса (другой порядок float-операций), поэтому torch-инференс
с GPU-весами на CPU-маке слегка расходился бы. Это НЕ проблема, потому что **боевой
инференс на маке/Render пойдёт через ONNX** (инкр.5) — ONNX даёт одинаковый вывод
на любом железе. До ONNX боевого torch-инференса на проде нет, так что обучаем на GPU.

Боевой `ml/`-код остаётся CPU-чистым (он нужен для ONNX-экспорта и локальных тестов).
GPU включается ТОЛЬКО в этом ноутбуке через патч ниже (ячейка 2b) — прод не трогаем.

## 1. Клонируем репу и добавляем её корень в `sys.path`

`iot_hub/__init__.py` отсутствует намеренно — пакет резолвится как implicit namespace
package (Python 3), ровно как в `tests/`. Достаточно положить корень репы в `sys.path`.

In [ ]:
import sys, os

REPO_URL = "https://github.com/vadim-white/IoT-hub-monitor.git"
REPO_DIR = "IoT-hub-monitor"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 $REPO_URL

repo_root = os.path.abspath(REPO_DIR)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("repo_root:", repo_root)

## 2. Фиксируем версии зависимостей

Версии **из `requirements.txt`** — критично для воспроизводимости весов. Colab несёт свои
torch/numpy; без пина веса могут чуть отличаться. Django НЕ ставим — `ml/`-ядро от него
не зависит (`loader.py` с ORM в Colab не используем, ряд собираем из CSV).

In [ ]:
# torch с CUDA (дефолтная сборка Colab уже GPU-enabled). statsmodels нужен для
# внутреннего Holt-Winters в lstm_lf_resid. numpy/sklearn пинуем для стабильности.
!pip install -q torch==2.2.2 numpy==1.26.3 scikit-learn==1.4.2 statsmodels==0.14.2

import torch
print("CUDA доступна:", torch.cuda.is_available(),
      "| устройство:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
# если False — Runtime → Change runtime type → T4 GPU, затем перезапустить ячейки

## 2b. GPU-патч для обучения lstm_lf_resid (только в этом ноутбуке)

Боевой класс `LSTMLevelFixResidualForecaster` считает на CPU (тензоры без `.to(device)`),
чтобы прод оставался CPU-чистым. Здесь, **не трогая репозиторий**, переопределяем три
метода так, чтобы обучение шло на GPU:

- `_build_and_train` — сеть и обучающие тензоры на `cuda`;
- `forecast` — вход на `cuda`, выход через `.cpu().numpy()`;
- `_dump_state` — веса сохраняются как **CPU**-state_dict, чтобы `.pt` грузился на
  маке/Render без GPU (иначе `torch.load` искал бы cuda-устройство).

Логика модели (level-fix на остатке HW) идентична боевой — меняется только устройство.

In [ ]:
# GPU-патч: переопределяем методы lstm_lf_resid на cuda (репозиторий не меняем)
import numpy as np, torch
from iot_hub.apps.telemetry.ml.forecasters.lstm_level_fix_residual import (
    LSTMLevelFixResidualForecaster as LF)
from iot_hub.apps.telemetry.ml.forecast_base import ForecastResult, future_timestamps
from iot_hub.apps.telemetry.ml.torch_utils import build_lstm_seq2seq, set_torch_determinism

DEVICE_T = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("обучение на:", DEVICE_T)

def _build_and_train_gpu(self, horizon):
    set_torch_determinism(self.random_state)
    r, w, std = self._resid, self.window, self._std
    n = len(r)
    X, D = [], []
    for t in range(w, n - horizon + 1):
        level = r[t - 1]
        X.append((r[t - w:t] - level) / std)
        D.append((r[t:t + horizon] - level) / std)
    if not X:
        self._model = None
        return
    X = torch.tensor(np.array(X), dtype=torch.float32).unsqueeze(-1).to(DEVICE_T)
    D = torch.tensor(np.array(D), dtype=torch.float32).to(DEVICE_T)
    model = build_lstm_seq2seq(self.hidden, horizon).to(DEVICE_T)
    opt = torch.optim.Adam(model.parameters(), lr=self.lr)
    loss_fn = torch.nn.MSELoss()
    model.train()
    for _ in range(self.epochs):
        opt.zero_grad(); loss = loss_fn(model(X), D); loss.backward(); opt.step()
    model.eval()
    self._model = model

def _forecast_gpu(self, horizon):
    if self._model is None or self._horizon != horizon:
        self._horizon = horizon
        self._build_and_train(horizon)
    hw_fc = np.asarray(self._hw.forecast(horizon).mean, dtype=float)
    if self._model is None:
        mean = hw_fc
    else:
        x = torch.tensor(self._tail, dtype=torch.float32).reshape(1, self.window, 1).to(DEVICE_T)
        with torch.no_grad():
            d = self._model(x).cpu().numpy().ravel()   # cuda → cpu перед numpy
        resid_fc = self._r_level + d * self._std
        mean = hw_fc + resid_fc[:horizon]
    ts = future_timestamps(self._last_ts, self._step, horizon)
    return ForecastResult(timestamps=ts, mean=mean, lower=None, upper=None, horizon=horizon,
        meta={"method": self.name, "seasonal_periods": self.seasonal_periods,
              "params": {"window": self.window, "epochs": self.epochs,
                         "hidden": self.hidden, "random_state": self.random_state}})

_orig_dump = LF._dump_state
def _dump_state_cpu(self, stem):
    # веса на cuda → перенести модель на cpu перед сохранением, чтобы .pt грузился
    # на маке/Render без GPU; затем вернуть на DEVICE_T (вдруг ещё нужен forecast)
    if self._model is not None:
        self._model.to("cpu")
    _orig_dump(self, stem)
    if self._model is not None:
        self._model.to(DEVICE_T)

LF._build_and_train = _build_and_train_gpu
LF.forecast = _forecast_gpu
LF._dump_state = _dump_state_cpu
print("патч применён: lstm_lf_resid обучается на", DEVICE_T)

## 3. Загружаем CSV и собираем `TimeSeries`

`load_series_from_csv` — офлайн-зеркало `loader.load_series` (тот же `TimeSeries`,
те же метки). Загрузите CSV через виджет ниже (или смонтируйте Google Drive).

In [ ]:
from google.colab import files
uploaded = files.upload()          # выберите train.csv из export_training_data
CSV_PATH = next(iter(uploaded))

# альтернатива — Google Drive:
# from google.colab import drive; drive.mount('/content/drive')
# CSV_PATH = '/content/drive/MyDrive/train.csv'
print("CSV:", CSV_PATH)

In [ ]:
from iot_hub.apps.telemetry.ml.dataset_csv import load_series_from_csv

DEVICE = "TEMP-001"        # serial_number, как в колонке CSV
METRIC = "temperature"     # metric_type

series = load_series_from_csv(CSV_PATH, DEVICE, METRIC)
print(f"точек: {len(series)}, аномалий: {int(series.labels.sum())}")

## 4. Обучаем модель

Гиперпараметры собираем через `cli_params` — тот же источник, что у команд
`train_models`/`detect_anomalies`. Это гарантирует, что **sha манифеста совпадёт** и
локальный `detect_anomalies --use-cache` примет веса без `CacheMismatchError`.

Меняйте `METHOD`/`opts` под нужную модель (`autoencoder` детектор, `lstm` форкастер).

In [ ]:
from iot_hub.apps.telemetry.ml.detectors import build_detector
from iot_hub.apps.telemetry.ml.cli_params import detector_params
import time

METHOD = "autoencoder"
# те же дефолты, что у train_models — менять ЗДЕСЬ, потом теми же значениями звать --use-cache
opts = {
    "window": 24, "threshold": 3.0,
    "epochs": 150, "latent_dim": 8, "lr": 1e-3,
    "threshold_percentile": 98.0, "random_state": 42,
    "contamination": 0.02, "n_estimators": 200,
}

model = build_detector(METHOD, **detector_params(METHOD, opts))
t0 = time.perf_counter()
model.fit(series)
print(f"{model.name} обучен за {time.perf_counter() - t0:.2f}с")

## 4b. (альтернатива) Обучаем форкастер `lstm_lf_resid` — победитель Фазы 5

Гибрид Holt-Winters + level-fix LSTM на остатке (раунд 4). `statsmodels` (внутренний
HW) ставится в ячейке версий. **Сначала выполните GPU-патч (ячейка 2b)** — иначе
обучение пойдёт на CPU.

Гиперпараметры **зафиксированы** в `cli_params.LSTM_LF_RESID_DEFAULTS` (window=144,
epochs=100, hidden=32) и НЕ берутся из общих `--window/--epochs` команд — поэтому
sha артефакта стабилен и локальный `forecast_telemetry --use-cache` примет веса без
`CacheMismatchError`. Менять гиперпараметры здесь не нужно (это «модель победителя»).

**Важно:** сеть LSTM строится лениво на первом `forecast(HORIZON)`. Чтобы веса (`.pt`)
обучились в Colab, а не локально, ниже вызываем `forecast()` ДО `save()` — с тем же
`HORIZON`, что используется в проде (по умолчанию 36). После переноса локальный
`forecast --use-cache` с тем же горизонтом загрузит сеть, а не переобучит.</cell>

In [ ]:
from iot_hub.apps.telemetry.ml.forecasters import build_forecaster
from iot_hub.apps.telemetry.ml.cli_params import forecaster_params
from iot_hub.apps.telemetry.ml.persistence import model_key
import time

METHOD = "lstm_lf_resid"
HORIZON = 36  # тот же горизонт, что у forecast_telemetry по умолчанию
# гиперпараметры фиксированы в LSTM_LF_RESID_DEFAULTS; из CLI уважается только random_state
model = build_forecaster(METHOD, **forecaster_params(METHOD, {"random_state": 42}))

t0 = time.perf_counter()
model.fit(series)                 # обучает внутренний HW + готовит остаток
model.forecast(HORIZON)           # ВАЖНО: строит и обучает LSTM-сеть здесь, в Colab
print(f"{model.name} обучен за {time.perf_counter() - t0:.2f}с")

stem = model_key(model.name, DEVICE, METRIC)
model.save(stem)                  # .npz + .pt (веса сети) + _hw.joblib + манифесты
print("артефакт:", stem.name)
!ls -la {stem.parent} | grep {model.name}

## 4c. (рекомендуется) Обучить все 4 temperature-устройства за один прогон

Цикл по TEMP-001/004/006/008 — те же, что прогоняются на Render. Для каждого:
загрузить свой CSV, `fit` → `forecast(HORIZON)` (строит и обучает сеть в Colab!) →
`save`. Гиперпараметры фиксированы в `LSTM_LF_RESID_DEFAULTS` — не трогаем.

**Перед запуском:** выполните GPU-патч (ячейка 2b), затем загрузите 4 файла `train_TEMP-001.csv … train_TEMP-008.csv`
(виджет ниже) или смонтируйте Drive. Имена должны совпадать с шаблоном `CSV_TMPL`.

In [ ]:
from google.colab import files
from iot_hub.apps.telemetry.ml.dataset_csv import load_series_from_csv
from iot_hub.apps.telemetry.ml.forecasters import build_forecaster
from iot_hub.apps.telemetry.ml.cli_params import forecaster_params
from iot_hub.apps.telemetry.ml.persistence import model_key
import time

DEVICES = ["TEMP-001", "TEMP-004", "TEMP-006", "TEMP-008"]
METRIC  = "temperature"
HORIZON = 36
CSV_TMPL = "train_{dev}.csv"   # шаблон имени выгруженного CSV (export_training_data)

uploaded = files.upload()      # выберите все 4 файла train_TEMP-XXX.csv сразу
print("загружено:", list(uploaded))

for dev in DEVICES:
    csv = CSV_TMPL.format(dev=dev)
    if csv not in uploaded:
        print(f"  {dev}: пропуск — нет {csv}"); continue
    series = load_series_from_csv(csv, dev, METRIC)
    model = build_forecaster("lstm_lf_resid", **forecaster_params("lstm_lf_resid", {}))
    t0 = time.perf_counter()
    model.fit(series)
    model.forecast(HORIZON)    # ВАЖНО: строит и обучает LSTM-сеть здесь, в Colab
    stem = model_key(model.name, dev, METRIC)
    model.save(stem)           # .npz + .pt + _hw.joblib + манифесты
    print(f"  {dev}: обучен за {time.perf_counter()-t0:.1f}с → {stem.name}")

print("\nГотово. Скачайте zip последней ячейкой (## 6).")

## 5. Сохраняем артефакт через тот же `save()`, что и прод

In [ ]:
from iot_hub.apps.telemetry.ml.persistence import model_key

stem = model_key(model.name, DEVICE, METRIC)   # ml/models/<name>__<device>__<metric>
model.save(stem)
print("артефакт:", stem.name)
!ls -la {stem.parent}

## 6. Скачиваем веса

Распакуйте архив локально в `iot_hub/apps/telemetry/ml/models/`, затем:
`python manage.py detect_anomalies --device TEMP-001 --metric temperature \`
`--method autoencoder --use-cache` — команда загрузит веса вместо переобучения.

In [ ]:
from google.colab import files

models_dir = stem.parent
!cd {repo_root}/iot_hub/apps/telemetry/ml && zip -r /content/ml_models.zip models
files.download("/content/ml_models.zip")